## **1. Projeto de Inteligência Artificial 2025/2026: Agentes para o Jogo PopOut**

**Faculdade de Ciências da Universidade do Porto (FCUP)** **Unidade Curricular:** Inteligência Artificial (CC2006)  
**Turma:** CC2006_PL1 | **Grupo:** 3  
**Docente:** Prof. Francesco Renna  

### **Equipe de Desenvolvimento**
| Número | Nome |
| :--- | :--- |
| **202300654** | Augusto Moreira |
| **202300276** | Guilherme Klippel |
| **202300916** | Yan Coelho |

---

### **Introdução e Contexto do Problema**

Este notebook documenta a conceção, implementação e análise de agentes de Inteligência Artificial para o jogo **PopOut**, uma variante estratégica e dinâmica do clássico *Connect-4* (Quatro em Linha). Tratando-se de um jogo de tabuleiro de soma nula e informação perfeita, o PopOut apresenta um desafio acrescido devido à sua mecânica de remoção de peças, que aumenta significativamente a complexidade e o fator de ramificação (*branching factor*) da árvore de jogo.

O objetivo deste projeto divide-se em duas abordagens fundamentais da IA:
1. **Procura Adversarial:** Implementação de um agente baseado em **Monte Carlo Tree Search (MCTS)** para atuar como um jogador de elite (*expert*).
2. **Aprendizagem Automática:** Utilização do MCTS para gerar conjuntos de dados de alta qualidade, servindo de base para o treino de um modelo de **Árvores de Decisão (algoritmo ID3)**, capaz de inferir regras táticas a partir das jogadas guardadas nos datasets.

### **Mecânicas e Regras Especiais**

A nossa implementação na classe base `PopOutGame` suporta o tabuleiro normal de 6x7 e introduz as seguintes ações:
* **Drop:** Colocação de uma peça no topo de uma coluna, caindo até à posição livre mais baixa.
* **PopOut:** Remoção de uma peça do próprio jogador na base (linha inferior) de uma coluna, fazendo com que todas as peças acima desçam uma posição.

Para garantir a robustez das simulações, foram rigorosamente implementadas as **três regras especiais** exigidas no guião:
* **Regra 1 (Vitória Simultânea):** Se uma jogada *PopOut* alinhar simultaneamente 4 peças para ambos os jogadores, a vitória é atribuída ao jogador que efetuou o movimento.
* **Regra 2 (Tabuleiro Cheio):** Caso a linha superior do tabuleiro esteja totalmente preenchida, o movimento *Drop* torna-se inválido, permitindo ao jogador escolher entre executar um *PopOut* válido ou declarar empate.
* **Regra 3 (Empate por Repetição):** O sistema monitoriza o histórico de estados (através de *hashes* do tabuleiro). Se o mesmo estado se repetir 3 vezes durante a partida, o jogo é declarado como empate.

---

### **Nota sobre a Estrutura deste Notebook**

Além de documentar a implementação, esta versão procura explicar as escolhas feitas ao longo do projeto: quando usamos MCTS puro ou otimizado, por que normalizamos o tabuleiro para a perspetiva do jogador atual, como tratamos conflitos no dataset e por que avaliamos os agentes tanto por métricas supervisionadas como por partidas reais.


## 1. Objetivos do Trabalho

A proposta do trabalho define dois objetivos técnicos principais:

1. Implementar um agente adversarial para PopOut usando **MCTS com UCT**.
2. Implementar uma **árvore de decisão ID3** sem recorrer a `scikit-learn` ou bibliotecas equivalentes para treino automático.

Além disso, o projeto deve suportar três cenários de jogo: humano vs humano, humano vs computador e computador vs computador. A árvore de decisão é avaliada primeiro com o dataset **Iris** e depois aplicada ao PopOut usando datasets gerados pelo MCTS.

**Decisão de organização:** o notebook segue a ordem das dependências do projeto. Primeiro validamos o jogo, depois o MCTS, depois os dados produzidos pelo MCTS, e só então treinamos a árvore. Esta ordem evita apresentar resultados de aprendizagem antes de explicar de onde vêm os rótulos e que garantias temos sobre as regras do jogo.

## 2. Preparação do Ambiente

Esta secção centraliza os imports, caminhos e funções auxiliares usadas ao longo do notebook.

**Decisão de reprodutibilidade:** as simulações mais pesadas não são executadas automaticamente. O notebook carrega datasets e modelos já gerados, permitindo que a correção seja rápida e consistente. Quando for necessário repetir experiências, as células correspondentes podem ser ativadas explicitamente.

**Decisão técnica:** usamos bibliotecas externas para manipulação de dados e visualização (`pandas`, `numpy`, `matplotlib`, `seaborn`, `networkx`), mas não para implementar automaticamente o MCTS ou a árvore ID3. Isto respeita a restrição da proposta: o algoritmo de decisão deve ser implementado pelo grupo.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
import os
import pickle
import time
import itertools
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.game import PopOutGame, PLAYER1, PLAYER2, _get_ai_move
from src.mcts import MCTS
from src.dataset_generator import run_batch_simulation
from src.decision_tree import (
    DecisionTreeID3,
    clean_conflicting_data,
    discretizar_largura_igual,
    discretizar_frequencia_igual,
    calcular_metricas,
    plotar_arvore_decisao,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd()
DATASETS_DIR = PROJECT_ROOT / "datasets"
RESULTS_DIR = PROJECT_ROOT / "resultados"
MODELS_DIR = PROJECT_ROOT / "modelos_treinados"

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("Projeto:", PROJECT_ROOT)
print("Datasets disponíveis:", len(list(DATASETS_DIR.glob("*.csv"))))
print("Modelos disponíveis:", len(list(MODELS_DIR.glob("*.pkl"))))

## 3. Regras e Representação do PopOut

O PopOut começa como o Connect-4 tradicional: os jogadores alternam turnos inserindo peças no topo de uma coluna. A diferença principal é a jogada **pop**, em que um jogador pode remover uma peça sua da base de uma coluna, fazendo as peças acima descerem.

A implementação usa uma matriz `6 x 7`:

- `0`: célula vazia;
- `1`: jogador 1;
- `2`: jogador 2.

As regras especiais consideradas são:

- se um pop cria quatro-em-linha para ambos os jogadores, vence quem fez o pop;
- quando o tabuleiro está cheio, o jogador pode declarar empate ou fazer pop;
- se o mesmo estado se repetir três vezes, pode ser declarado empate.

**Decisão de representação:** optámos por uma matriz NumPy porque o tabuleiro tem dimensão fixa e as operações de leitura por linha, coluna e diagonal são frequentes. A matriz também facilita gerar os vetores `pos_0` a `pos_41` usados posteriormente no dataset.

**Decisão de regras:** a verificação de vitória está no motor do jogo e não no agente. Assim, MCTS, árvore de decisão e interface usam a mesma fonte de verdade. Isto reduz o risco de cada agente interpretar o jogo de forma diferente.

In [ ]:
game = PopOutGame()
print("Tabuleiro inicial:")
print(game.board)
print("Jogadas válidas no início:", game.get_valid_moves())

# Pequena demonstração de drop e pop
game.drop_piece(3, PLAYER1)
game.current_player = PLAYER2
game.drop_piece(3, PLAYER2)
game.current_player = PLAYER1
print("\nDepois de duas jogadas na coluna 3:")
print(game.board)
print("Jogadas válidas para o jogador 1:", game.get_valid_moves())

## 4. Arquitetura da Solução

A solução está separada em módulos:

| Ficheiro | Responsabilidade |
|---|---|
| `src/game.py` | Motor de regras, validação de jogadas e modos de jogo. |
| `src/mcts.py` | Implementação do agente MCTS com UCT. |
| `src/dataset_generator.py` | Simulação IA vs IA e exportação de datasets. |
| `src/decision_tree.py` | Implementação do ID3, discretização, métricas e visualização. |
| `play.py` | Interface de terminal para jogar ou observar partidas. |

**Decisão de arquitetura:** o código foi separado por responsabilidade para que o notebook seja uma camada de experiência e análise, não o único local onde a solução existe. Isto permite usar os mesmos módulos para jogar, gerar dados, treinar modelos e comparar agentes.

**Decisão de manutenção:** manter o motor do jogo independente do MCTS e do ID3 torna mais fácil alterar um agente sem mudar as regras. Também permite comparar agentes diferentes de forma justa, porque todos jogam sobre a mesma implementação de PopOut.

## 5. Monte Carlo Tree Search

O MCTS implementado segue as quatro fases clássicas:

1. **Selection:** percorre a árvore usando UCT/UCB.
2. **Expansion:** adiciona um novo filho a partir de uma jogada ainda não explorada.
3. **Simulation:** joga até estado terminal ou profundidade limite.
4. **Backpropagation:** propaga o resultado pelos nós visitados.

A função UCT equilibra exploração e aproveitamento. O parâmetro `c` controla esse equilíbrio: valores maiores favorecem exploração; valores menores favorecem ramos já promissores.

**Decisão importante:** o agente principal de MCTS **não é totalmente puro**. Além da versão pura, foi implementada uma versão otimizada com duas heurísticas simples antes/durante as simulações:

- se existir uma jogada vencedora imediata, ela é escolhida;
- se o adversário tiver uma vitória imediata por `drop`, o agente tenta bloquear.

Esta escolha foi feita porque o PopOut tem muitos estados táticos de curto prazo. Um MCTS puramente aleatório pode gastar várias iterações para descobrir algo óbvio, como uma vitória em uma jogada. A heurística reduz esse desperdício e torna o agente mais forte com o mesmo orçamento computacional.

**Compromisso assumido:** a heurística aproxima o MCTS de um agente híbrido. Isto melhora a qualidade prática das jogadas, mas significa que os resultados do agente otimizado não medem apenas o MCTS puro. Por isso mantemos também uma configuração `pure_mode=True`, usada como linha de base para comparação.

In [ ]:
mcts_configs = {
    "MCTS Base": MCTS(iterations=1000, c=1.41, max_children=None, max_depth=None, pure_mode=False),
    "MCTS Pure": MCTS(iterations=1000, c=1.41, max_children=None, max_depth=None, pure_mode=True),
    "MCTS Exploratório": MCTS(iterations=1000, c=3.0, max_children=None, max_depth=None, pure_mode=False),
    "MCTS Filhos Limitados": MCTS(iterations=1000, c=1.41, max_children=3, max_depth=None, pure_mode=False),
}

pd.DataFrame([
    {
        "Agente": nome,
        "Iterações": agente.iterations,
        "C": agente.c,
        "Max Children": "All" if agente.max_children is None else agente.max_children,
        "Max Depth": "None" if agente.max_depth is None else agente.max_depth,
        "Pure Mode": agente.pure_mode,
    }
    for nome, agente in mcts_configs.items()
])

In [ ]:
# Demonstração curta: poucas iterações para manter a célula rápida.
demo_game = PopOutGame()
demo_agent = MCTS(iterations=50, c=1.41, max_children=None, max_depth=30, pure_mode=False)
move = demo_agent.search(demo_game)
print("Jogada escolhida pelo MCTS no tabuleiro inicial:", move)

## 6. Geração do Dataset PopOut

Os datasets de PopOut são gerados por partidas automáticas entre agentes MCTS. Cada linha representa uma decisão tomada durante uma partida.

Colunas principais:

| Coluna | Significado |
|---|---|
| `game_id` | Identificador da partida. |
| `pos_0` a `pos_41` | Estado do tabuleiro em formato vetorial. |
| `p_turn` | Jogador que vai decidir a jogada. |
| `col` | Coluna escolhida. |
| `type` | Tipo da jogada: `d` para drop, `p` para pop. |
| `winner` | Vencedor final da partida, ou `0` em caso de empate. |

**Decisão de geração:** usamos MCTS vs MCTS porque o segundo dataset pedido não existe pronto na plataforma. O MCTS funciona como gerador de rótulos: para cada estado observado, a jogada escolhida pelo agente torna-se o alvo que a árvore tenta aprender.

**Decisão de rastreabilidade:** guardamos `game_id`, `p_turn` e `winner` mesmo que nem todas essas colunas sejam usadas diretamente no treino. Elas permitem filtrar jogadas de vencedores, estudar empates e auditar se um exemplo veio de uma partida bem-sucedida ou não.

**Compromisso assumido:** o dataset não representa jogadas perfeitas; representa jogadas escolhidas pelo MCTS configurado. Portanto, a árvore aprende a imitar esse agente, não a resolver PopOut de forma ótima.

In [ ]:
RUN_DATASET_GENERATION = False

if RUN_DATASET_GENERATION:
    ia_1 = MCTS(iterations=100, c=1.41, max_children=None, max_depth=None, pure_mode=True)
    ia_2 = MCTS(iterations=100, c=1.41, max_children=None, max_depth=None, pure_mode=True)
    run_batch_simulation(num_games=50, ia_1=ia_1, ia_2=ia_2)
else:
    print("Geração de novos dados não executada nesta célula. A usar datasets existentes em", DATASETS_DIR)

In [ ]:
dataset_files = sorted(DATASETS_DIR.glob("P1_*.csv"))
summary_rows = []
for path in dataset_files:
    df_tmp = pd.read_csv(path)
    summary_rows.append({
        "Dataset": path.name,
        "Linhas": len(df_tmp),
        "Jogos": df_tmp["game_id"].nunique() if "game_id" in df_tmp else np.nan,
        "Empates": int((df_tmp["winner"] == 0).sum()) if "winner" in df_tmp else np.nan,
    })

pd.DataFrame(summary_rows).sort_values("Linhas", ascending=False).head(10)

## 7. Análise Exploratória dos Dados

Antes de treinar uma árvore de decisão, é importante verificar se os dados têm coerência estratégica. Nesta análise observamos vencedores, tipos de jogadas, distribuição por coluna e duração das partidas.

**Decisão de avaliação:** não treinamos a árvore diretamente sem olhar para o dataset, porque o ID3 é sensível a ruído e conflitos. Se o dataset tiver muitas decisões contraditórias para estados iguais, a árvore fica maior, menos interpretável e potencialmente menos útil.

**O que procuramos nesta secção:**

- se há excesso de empates;
- se as jogadas se concentram em colunas plausíveis;
- se `drop` e `pop` aparecem em proporções razoáveis;
- se a duração das partidas sugere jogos muito curtos, muito longos ou ciclos repetitivos.

In [ ]:
POPOUT_DATASET = DATASETS_DIR / "P1_it1000_c1.41_mcAll_dNone_vs_P2_it1000_c1.41_mcAll_dNone.csv"

df_popout = pd.read_csv(POPOUT_DATASET)
df_popout["target_move"] = df_popout["col"].astype(str) + "_" + df_popout["type"]

print("Dataset:", POPOUT_DATASET.name)
print("Dimensão:", df_popout.shape)
df_popout.head()

In [ ]:
def classificar_status_jogada(row):
    if row["winner"] == 0:
        return "Empate"
    if row["p_turn"] == row["winner"]:
        return "Vencedor"
    return "Perdedor"

eda_df = df_popout.copy()
eda_df["status"] = eda_df.apply(classificar_status_jogada, axis=1)
eda_games = eda_df.drop_duplicates(subset=["game_id"])[["game_id", "winner"]]
status_palette = {"Vencedor": "#2ecc71", "Perdedor": "#e67e22", "Empate": "#95a5a6"}

fig, axes = plt.subplots(4, 2, figsize=(18, 22))
fig.suptitle(
    f"Análise global e estratégica: {POPOUT_DATASET.name}\n"
    f"Total de partidas: {eda_games['game_id'].nunique()}",
    fontsize=18,
    fontweight="bold",
    y=0.99,
)

sns.countplot(data=eda_games, x="winner", hue="winner", order=[0, 1, 2],
              ax=axes[0, 0], palette=["#95a5a6", "#3498db", "#e74c3c"], legend=False)
axes[0, 0].set_title("1. Resultados finais das partidas")
axes[0, 0].set_xticklabels(["Empate", "Vitória P1", "Vitória P2"])
axes[0, 0].set_xlabel("Resultado")
axes[0, 0].set_ylabel("Partidas")

sns.countplot(data=eda_df, x="p_turn", hue="p_turn", ax=axes[0, 1],
              palette=["#3498db", "#e74c3c"], legend=False)
axes[0, 1].set_title("2. Volume total de jogadas por jogador")
axes[0, 1].set_xlabel("Jogador")
axes[0, 1].set_ylabel("Jogadas")

sns.countplot(data=eda_df, x="col", hue="p_turn", ax=axes[1, 0], palette=["#3498db", "#e74c3c"])
axes[1, 0].set_title("3. Distribuição de colunas por jogador")
axes[1, 0].set_xlabel("Coluna")
axes[1, 0].set_ylabel("Jogadas")

move_types = eda_df["type"].value_counts()
axes[1, 1].pie(move_types, labels=move_types.index, autopct="%1.1f%%",
               colors=["#4C72B0", "#DD8452"], startangle=90)
axes[1, 1].set_title("4. Proporção global: drop vs pop")

sns.countplot(data=eda_df, x="col", hue="status", ax=axes[2, 0], palette=status_palette)
axes[2, 0].set_title("5. Colunas por status final da jogada")
axes[2, 0].set_xlabel("Coluna")
axes[2, 0].set_ylabel("Jogadas")

sns.countplot(data=eda_df, x="type", hue="status", ax=axes[2, 1], palette=status_palette)
axes[2, 1].set_title("6. Uso de drop/pop por status")
axes[2, 1].set_xlabel("Tipo de jogada")
axes[2, 1].set_ylabel("Jogadas")

board_cols = [f"pos_{i}" for i in range(42)]
occupancy = (eda_df[board_cols] > 0).mean().values.reshape(6, 7)
sns.heatmap(occupancy, annot=True, fmt=".2f", cmap="YlOrRd", ax=axes[3, 0], cbar=False)
axes[3, 0].set_title("7. Mapa médio de ocupação")
axes[3, 0].set_xlabel("Coluna")
axes[3, 0].set_ylabel("Linha")

pop_moves = eda_df[eda_df["type"] == "p"]
if not pop_moves.empty:
    sns.countplot(data=pop_moves, x="col", hue="status", ax=axes[3, 1], palette=status_palette)
    axes[3, 1].set_title("8. Zonas de PopOut por status")
    axes[3, 1].set_xlabel("Coluna")
    axes[3, 1].set_ylabel("PopOuts")
else:
    axes[3, 1].text(0.5, 0.5, "Sem jogadas PopOut", ha="center", va="center")
    axes[3, 1].set_axis_off()

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


## 8. Preparação do Dataset para a árvore PopOut

Para a árvore de decisão, o tabuleiro é convertido para uma perspetiva relativa ao jogador da vez:

- `1`: peça do jogador que vai jogar;
- `-1`: peça do adversário;
- `0`: célula vazia.

Isto evita treinar regras separadas para jogador 1 e jogador 2. A jogada alvo é representada por `coluna_tipo`, por exemplo `3_d` ou `2_p`.

**Decisão de normalização:** sem esta transformação, a árvore teria de aprender duas políticas equivalentes: uma para o jogador 1 e outra para o jogador 2. Ao converter tudo para “eu” contra “adversário”, duplicações estratégicas são reduzidas e o modelo tende a generalizar melhor.

**Decisão sobre conflitos:** o mesmo estado pode aparecer mais de uma vez com jogadas diferentes, principalmente porque o MCTS tem aleatoriedade e pode escolher alternativas próximas. Para tornar o ID3 treinável, agrupamos estados iguais e preservamos a jogada mais frequente. Esta escolha sacrifica algumas alternativas válidas, mas reduz ambiguidade no alvo.

**Decisão sobre `only_winners`:** mantemos duas formas de treinar. Com todas as jogadas, a árvore aprende o comportamento geral do MCTS. Com apenas jogadas feitas por jogadores que venceram, a árvore tenta focar exemplos associados a partidas bem-sucedidas. Nenhuma das duas é universalmente melhor; por isso ambas são comparadas.

In [ ]:
def preparar_dataset_popout(caminho_dataset, only_winners=False):
    df = pd.read_csv(caminho_dataset).copy()

    if only_winners:
        df = df[(df["winner"] != 0) & (df["p_turn"] == df["winner"])]

    colunas_tabuleiro = [f"pos_{i}" for i in range(42)]
    for col in colunas_tabuleiro:
        df[col] = np.where(df[col] == 0, 0, np.where(df[col] == df["p_turn"], 1, -1))

    df["target_move"] = df["col"].astype(str) + "_" + df["type"]
    df = df.drop(columns=["game_id", "winner", "col", "type", "p_turn"])
    df = clean_conflicting_data(df, target_name="target_move")
    return df


def calcular_profundidade(node):
    if node is None or not getattr(node, "children", None):
        return 0
    return 1 + max(calcular_profundidade(child) for child in node.children.values())


def stratified_split(df, target_col, test_frac=0.3, seed=42):
    test = df.groupby(target_col, group_keys=False).sample(frac=test_frac, random_state=seed)
    train = df.drop(test.index)
    return train.reset_index(drop=True), test.reset_index(drop=True)


df_tree_all = preparar_dataset_popout(POPOUT_DATASET, only_winners=False)
df_tree_win = preparar_dataset_popout(POPOUT_DATASET, only_winners=True)

print("Dataset para árvore com todas as jogadas:", df_tree_all.shape)
print("Dataset para árvore só com jogadas vencedoras:", df_tree_win.shape)
df_tree_all.head()

## 9. Implementação do ID3

A árvore de decisão foi implementada de raiz em `src/decision_tree.py`. A implementação calcula:

- entropia de Shannon;
- ganho de informação;
- escolha recursiva do melhor atributo;
- nós folha por classe pura, falta de atributos ou profundidade máxima;
- fallback para a classe maioritária quando surge um valor não observado no treino.

Não é usado `scikit-learn` para criar ou treinar árvores de decisão.

**Decisão algorítmica:** ID3 foi escolhido porque é o procedimento pedido na proposta e porque produz uma estrutura interpretável. A interpretabilidade é relevante aqui: além de jogar, queremos observar que atributos do estado influenciam a jogada prevista.

**Compromisso assumido:** ID3 clássico trabalha melhor com atributos categóricos. No Iris isso exige discretização. No PopOut, os atributos já são discretos (`-1`, `0`, `1`), mas existem muitos atributos e muitas combinações possíveis, o que pode gerar árvores grandes.

**Decisão de fallback:** quando a árvore encontra um valor não visto no treino, devolve a classe maioritária do nó. Esta escolha evita falhas em tempo de jogo e dá ao agente uma resposta válida mesmo em estados fora da distribuição de treino.

In [ ]:
# Demonstração pequena e controlada do ID3.
df_demo = pd.DataFrame({
    "ceu": ["sol", "sol", "chuva", "chuva"],
    "vento": ["fraco", "forte", "fraco", "forte"],
    "target": ["jogar", "nao_jogar", "jogar", "nao_jogar"],
})

arvore_demo = DecisionTreeID3()
arvore_demo.fit(df_demo, target_name="target")
print("Previsões:", arvore_demo.predict(df_demo.drop(columns=["target"])))

## 10. Validação com o Dataset Iris

O Iris funciona como teste inicial da implementação ID3. Como os atributos são numéricos, a proposta exige uma estratégia de discretização. Aqui comparamos três configurações:

1. valores originais;
2. discretização por largura igual;
3. discretização por frequência igual.

**Decisão experimental:** testamos primeiro com Iris porque é um problema conhecido e pequeno. Se a árvore falhar aqui, o erro provavelmente está no ID3 ou na discretização, não no dataset PopOut.

**Decisão sobre discretização:** usamos duas estratégias simples e explicáveis. A discretização por largura igual divide o intervalo numérico em faixas do mesmo tamanho. A discretização por frequência igual tenta equilibrar o número de exemplos em cada faixa. A comparação mostra o impacto da preparação dos dados sobre a complexidade da árvore.

In [ ]:
df_iris = pd.read_csv(DATASETS_DIR / "iris.csv")
feature_cols = ["sepallength", "sepalwidth", "petallength", "petalwidth"]

def avaliar_iris(df_entrada, descricao):
    df_model = df_entrada.drop(columns=["ID"], errors="ignore").copy()
    train, test = stratified_split(df_model, "class", test_frac=0.3, seed=RANDOM_STATE)
    tree = DecisionTreeID3()
    tree.fit(train, target_name="class")
    metricas_train = calcular_metricas(tree, train, "class")
    metricas_test = calcular_metricas(tree, test, "class")
    return {
        "Experiência": descricao,
        "Treino": len(train),
        "Teste": len(test),
        "Profundidade": calcular_profundidade(tree.root),
        "Acurácia Treino": metricas_train["acuracia"],
        "Acurácia Teste": metricas_test["acuracia"],
    }, tree

iris_original = df_iris.copy()
iris_width = discretizar_largura_igual(df_iris, feature_cols)
iris_freq = discretizar_frequencia_igual(df_iris, feature_cols)

resultados_iris = []
arvores_iris = {}
for descricao, df_exp in [
    ("Original", iris_original),
    ("Largura Igual", iris_width),
    ("Frequência Igual", iris_freq),
]:
    resultado, arvore = avaliar_iris(df_exp, descricao)
    resultados_iris.append(resultado)
    arvores_iris[descricao] = arvore

pd.DataFrame(resultados_iris)

### Discussão do Iris

A discretização é importante porque o ID3 trabalha naturalmente com atributos categóricos. Quando se usam valores numéricos crus, a árvore tende a criar ramos muito específicos e menos interpretáveis. Com discretização, a árvore fica mais compacta e a decisão passa a usar intervalos de valores.

**Interpretação esperada:** uma árvore com 100% de acerto no treino, mas com profundidade elevada, pode estar apenas a memorizar exemplos. Neste projeto, uma árvore ligeiramente menos perfeita, mas menor e mais interpretável, pode ser preferível porque demonstra melhor a lógica de classificação.

### Visualização das árvores do Iris

Além da tabela de métricas, é útil visualizar as árvores obtidas no Iris. A árvore sem discretização tende a ser menos interpretável, enquanto as versões discretizadas mostram decisões por intervalos.

A visualização completa pode gerar figuras grandes. Por isso, a execução fica controlada por uma variável booleana, permitindo ativar as figuras apenas quando necessário para a apresentação.


In [ ]:
PLOT_IRIS_TREES = False

if PLOT_IRIS_TREES:
    for nome, arvore in arvores_iris.items():
        metricas_visuais = next(item for item in resultados_iris if item["Experiência"] == nome).copy()
        metricas_visuais = {
            "profundidade": metricas_visuais["Profundidade"],
            "total": metricas_visuais["Teste"],
            "corretas": "N/A",
            "acuracia": metricas_visuais["Acurácia Teste"],
        }
        plotar_arvore_decisao(
            arvore.root,
            titulo=f"árvore Iris: {nome}",
            metricas=metricas_visuais,
        )
else:
    print("Visualização completa das árvores Iris não executada nesta célula. Defina PLOT_IRIS_TREES = True para gerar as figuras.")


## 11. Treino da árvore para PopOut

A árvore PopOut aprende a imitar decisões produzidas pelo MCTS. Para controlar ambiguidade, estados repetidos com jogadas diferentes são agrupados e fica a jogada mais frequente.

Nesta versão do notebook, o treino completo fica disponível, mas os modelos já treinados são carregados da pasta `modelos_treinados` para acelerar a execução.

**Decisão prática:** treinar a árvore completa pode ser demorado e produzir modelos grandes. Para a entrega, é melhor carregar modelos persistidos e apresentar métricas sobre eles. Isto torna o notebook mais estável para o avaliador.

**Decisão de comparação:** avaliamos árvores treinadas com conjuntos diferentes (`all` e `win`) para perceber se filtrar apenas jogadas vencedoras melhora a política. Esta comparação é mais informativa do que apresentar apenas uma árvore final sem justificar a origem dos exemplos.

In [ ]:
RUN_POPOUT_TRAINING = False

if RUN_POPOUT_TRAINING:
    train_popout, test_popout = stratified_split(df_tree_all, "target_move", test_frac=0.25, seed=RANDOM_STATE)
    tree_popout = DecisionTreeID3(max_depth=100)
    tree_popout.fit(train_popout, target_name="target_move")
    print(calcular_metricas(tree_popout, test_popout, "target_move"))
else:
    print("Treino completo não executado nesta célula. A carregar modelos previamente treinados na próxima célula.")

### Treino, Persistência e Visualização da árvore PopOut

Além de carregar modelos já treinados, esta secção define funções para treinar, guardar e visualizar árvores completas. A execução fica controlada por variáveis booleanas para evitar sobrescrever modelos finais ou gerar figuras grandes sem intenção.

Estas funções são úteis para criar uma árvore final para submissão ou para inspecionar visualmente um modelo já guardado.


In [ ]:
def treinar_e_guardar_arvore_popout(caminho_dataset, nome_modelo, only_winners=False, max_depth=100):
    df_treino = preparar_dataset_popout(caminho_dataset, only_winners=only_winners)
    arvore = DecisionTreeID3(max_depth=max_depth)
    arvore.fit(df_treino, target_name="target_move")

    MODELS_DIR.mkdir(exist_ok=True)
    caminho_modelo = MODELS_DIR / f"{nome_modelo}.pkl"
    with open(caminho_modelo, "wb") as f:
        pickle.dump(arvore, f)

    print(f"Modelo guardado em: {caminho_modelo}")
    print(f"Amostras usadas: {len(df_treino)} | Profundidade: {calcular_profundidade(arvore.root)}")
    return arvore


def visualizar_arvore_guardada(nome_modelo, dataset_teste=None, only_winners=False):
    caminho_modelo = MODELS_DIR / f"{nome_modelo}.pkl"
    with open(caminho_modelo, "rb") as f:
        arvore = pickle.load(f)

    metricas = {
        "profundidade": calcular_profundidade(arvore.root),
        "total": "N/A",
        "corretas": "N/A",
        "acuracia": 0.0,
    }

    if dataset_teste is not None:
        df_teste = preparar_dataset_popout(dataset_teste, only_winners=only_winners)
        metricas_teste = calcular_metricas(arvore, df_teste, "target_move")
        metricas.update(metricas_teste)
        metricas["profundidade"] = calcular_profundidade(arvore.root)

    plotar_arvore_decisao(
        arvore.root,
        titulo=f"Estrutura da árvore: {nome_modelo}",
        metricas=metricas,
    )
    return arvore

RUN_FINAL_TREE_TRAINING = False
PLOT_SAVED_POPOUT_TREE = False

if RUN_FINAL_TREE_TRAINING:
    treinar_e_guardar_arvore_popout(
        caminho_dataset=POPOUT_DATASET,
        nome_modelo="tree_final_candidate",
        only_winners=False,
        max_depth=100,
    )

if PLOT_SAVED_POPOUT_TREE:
    visualizar_arvore_guardada(
        nome_modelo="best_tree_all",
        dataset_teste=POPOUT_DATASET,
        only_winners=False,
    )
else:
    print("Treino/visualização completa da árvore PopOut não executados nesta célula.")


In [ ]:
def carregar_arvores_modelos(pasta=MODELS_DIR):
    modelos = {}
    for path in sorted(Path(pasta).glob("*.pkl")):
        with open(path, "rb") as f:
            modelos[path.stem] = pickle.load(f)
    return modelos

arvores = carregar_arvores_modelos()
print("árvores carregadas:", list(arvores.keys()))

In [ ]:
def avaliar_modelo_popout(nome_modelo, modelo, dataset_path, only_winners=False):
    df_eval = preparar_dataset_popout(dataset_path, only_winners=only_winners)
    metricas = calcular_metricas(modelo, df_eval, "target_move")
    return {
        "Modelo": nome_modelo,
        "Dataset": Path(dataset_path).name,
        "Apenas Vencedores": only_winners,
        "Amostras": metricas["total"],
        "Corretas": metricas["corretas"],
        "Acurácia": metricas["acuracia"],
        "Profundidade": calcular_profundidade(modelo.root) if hasattr(modelo, "root") else np.nan,
    }

avaliacoes_popout = []
for nome, modelo in arvores.items():
    if nome in {"best_tree_all", "worst_tree_all"}:
        avaliacoes_popout.append(avaliar_modelo_popout(nome, modelo, POPOUT_DATASET, only_winners=False))
    if nome in {"best_tree_win", "worst_tree_win"}:
        avaliacoes_popout.append(avaliar_modelo_popout(nome, modelo, POPOUT_DATASET, only_winners=True))

pd.DataFrame(avaliacoes_popout).sort_values("Acurácia", ascending=False)

## 12. Benchmark entre Agentes

Para comparar os agentes, foram executados torneios automáticos entre combinações de MCTS e árvores de decisão. Os ficheiros em `resultados/` registam o vencedor, quem começou e o número total de jogadas.

**Decisão de avaliação:** accuracy da árvore não é suficiente. Uma árvore pode prever bem o dataset e ainda jogar mal se os erros acontecerem em estados críticos. Por isso, além das métricas supervisionadas, comparamos agentes em partidas reais.

**Decisão de alternância:** guardar `Quem_Comecou` é importante porque em jogos de tabuleiro o primeiro jogador pode ter vantagem. Sem essa coluna, uma diferença de vitórias poderia ser confundida com vantagem de início.

**Métrica principal:** usamos número de vitórias, empates e duração média das partidas. Estas métricas não provam jogo ótimo, mas permitem comparar comportamento relativo entre configurações.

In [ ]:
benchmark_files = sorted(RESULTS_DIR.glob("*.csv"))
benchmark_summary = []
for path in benchmark_files:
    df_b = pd.read_csv(path)
    if "Vencedor" not in df_b.columns:
        continue
    counts = df_b["Vencedor"].value_counts(dropna=False).to_dict()
    benchmark_summary.append({
        "Ficheiro": path.name,
        "Jogos": len(df_b),
        "Jogadas Médias": df_b["Total_Jogadas"].mean() if "Total_Jogadas" in df_b else np.nan,
        "Vencedor Mais Frequente": df_b["Vencedor"].mode()[0] if len(df_b) else None,
        "Distribuição": counts,
    })

pd.DataFrame(benchmark_summary)

In [ ]:
BENCHMARK_FILE = RESULTS_DIR / "benchmark_mcts1000_c1.41_mcAll_dnone_vs_mcts1000_c3_mcAll_dnone.csv"
df_bench = pd.read_csv(BENCHMARK_FILE)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.countplot(data=df_bench, x="Vencedor", ax=axes[0], palette="viridis")
axes[0].set_title("Vitórias por agente")
axes[0].tick_params(axis="x", rotation=20)

sns.histplot(data=df_bench, x="Total_Jogadas", bins=15, ax=axes[1], color="#55A868")
axes[1].set_title("Duração das partidas")
axes[1].set_xlabel("Total de jogadas")

plt.tight_layout()
plt.show()

df_bench["Vencedor"].value_counts(normalize=True).mul(100).round(2).rename("Percentagem")

### Síntese dos Modelos e Comparações Realizadas

Esta secção organiza os resultados que sustentam a discussão final do projeto. A ideia é separar claramente três níveis de comparação:

1. **Datasets gerados por MCTS:** variam iterações, constante `c`, número máximo de filhos, profundidade e modo puro/otimizado.
2. **Árvores ID3 treinadas:** comparam datasets fortes/fracos e treino com todas as jogadas vs apenas jogadas de vencedores.
3. **Benchmarks em partidas:** comparam agentes em jogos completos, que é uma avaliação mais realista do que apenas accuracy supervisionada.

Esta organização é importante porque o projeto não tem apenas um modelo final. O objetivo é mostrar que testámos alternativas e que as escolhas finais são justificadas pelos resultados.


In [ ]:
modelos_resumo = pd.DataFrame([
    {
        "Modelo": "best_tree_all",
        "Fonte do dataset": "MCTS forte / otimizado, alta qualidade",
        "Filtro de treino": "Todas as jogadas",
        "Ideia testada": "Aprender o comportamento geral do MCTS forte",
    },
    {
        "Modelo": "best_tree_win",
        "Fonte do dataset": "MCTS forte / otimizado, alta qualidade",
        "Filtro de treino": "Apenas jogadas do vencedor",
        "Ideia testada": "Reduzir ruído e focar decisões associadas a vitórias",
    },
    {
        "Modelo": "worst_tree_all",
        "Fonte do dataset": "MCTS mais fraco / puro, menor qualidade",
        "Filtro de treino": "Todas as jogadas",
        "Ideia testada": "Medir impacto de aprender com dados piores",
    },
    {
        "Modelo": "worst_tree_win",
        "Fonte do dataset": "MCTS mais fraco / puro, menor qualidade",
        "Filtro de treino": "Apenas jogadas do vencedor",
        "Ideia testada": "Ver se filtrar vencedores ajuda mesmo com dados fracos",
    },
])

modelos_resumo


### Hiperparâmetros Testados no MCTS

Os nomes dos datasets guardam os hiperparâmetros usados. Por exemplo:

`P1_it1000_c1.41_mcAll_dNone_vs_P2_it1000_c1.41_mcAll_dNone.csv`

significa que os dois jogadores usaram MCTS com `1000` iterações, constante `c=1.41`, todos os filhos permitidos (`mcAll`) e sem limite de profundidade (`dNone`).

As principais variáveis exploradas foram:

| Hiperparâmetro | Significado | Comparação esperada |
|---|---|---|
| `iterations` | Número de simulações MCTS por jogada | Mais iterações tende a melhorar qualidade, mas aumenta tempo. |
| `c` | Constante de exploração UCT | Controla exploração vs aproveitamento. |
| `max_children` | Número máximo de filhos expandidos por nó | Limitar filhos reduz espaço de busca, mas pode excluir boas jogadas. |
| `max_depth` | Profundidade máxima do rollout | Limitar profundidade reduz tempo, mas pode perder consequências futuras. |
| `pure_mode` | MCTS puro ou otimizado | O modo otimizado usa heurísticas de vitória/bloqueio imediato. |


In [ ]:
def resumir_dataset_mcts(path):
    df = pd.read_csv(path)
    return {
        "Dataset": path.name,
        "Linhas": len(df),
        "Partidas": df["game_id"].nunique(),
        "Vitórias P1": int((df["winner"] == 1).sum()),
        "Vitórias P2": int((df["winner"] == 2).sum()),
        "Empates": int((df["winner"] == 0).sum()),
    }

resumo_datasets_mcts = pd.DataFrame(
    resumir_dataset_mcts(path) for path in sorted(DATASETS_DIR.glob("P1_*.csv"))
)

resumo_datasets_mcts.sort_values(["Partidas", "Linhas"], ascending=False).head(12)


### Resultados dos Benchmarks Principais

Os ficheiros em `resultados/` registam partidas completas entre agentes. Cada linha representa uma partida, com o vencedor, quem começou e o número total de jogadas.

As comparações mais importantes para apresentar são:

- **MCTS otimizado vs MCTS puro:** mede o impacto das heurísticas táticas.
- **`max_children` limitado vs todos os filhos:** mede o custo de restringir a expansão da árvore.
- **profundidade limitada vs sem limite:** mede o trade-off entre velocidade e qualidade.
- **`c=1.41` vs `c=3`:** mede o impacto da constante de exploração.
- **árvore `all` vs árvore `win`:** mede se filtrar apenas jogadas vencedoras melhora a política aprendida.


In [ ]:
comparacoes_principais = {
    "MCTS otimizado vs MCTS puro": RESULTS_DIR / "P1_it1000_c1.41_mcAll_dNone_opt_vs_P2_it1000_c1.41_mcAll_dNone_pure.csv",
    "max_children=3 vs max_children=All": RESULTS_DIR / "P1_it1000_c1.41_mc3_dNone_vs_P2_it1000_c1.41_mcAll_dNone.csv",
    "max_depth=10 vs sem limite": RESULTS_DIR / "P1_it1000_c1.41_mcAll_d10_vs_P2_it1000_c1.41_mcAll_dNone.csv",
    "c=1.41 vs c=3": RESULTS_DIR / "benchmark_mcts1000_c1.41_mcAll_dnone_vs_mcts1000_c3_mcAll_dnone.csv",
    "best_tree_all vs best_tree_win": RESULTS_DIR / "benchmark_best_tree_all_vs_best_tree_win.csv",
}

linhas = []
for nome_comparacao, caminho in comparacoes_principais.items():
    if not caminho.exists():
        linhas.append({"Comparacao": nome_comparacao, "Observacao": "CSV nao encontrado"})
        continue

    df_comp = pd.read_csv(caminho)
    vencedores = df_comp["Vencedor"].value_counts().to_dict()
    linhas.append({
        "Comparacao": nome_comparacao,
        "Ficheiro": caminho.name,
        "Partidas": len(df_comp),
        "Vencedores": vencedores,
        "Jogadas medias": round(df_comp["Total_Jogadas"].mean(), 2),
    })

pd.DataFrame(linhas)


### Interpretação das Comparações

**MCTS otimizado vs MCTS puro.**  
A versão otimizada venceu a maioria das partidas contra a versão pura. Isto mostra que as heurísticas de vitória imediata e bloqueio de derrota iminente melhoram o desempenho prático. A desvantagem é que cada rollout fica mais caro, porque há mais verificações táticas durante a simulação.

**`max_children=3` vs `max_children=All`.**  
A configuração com todos os filhos teve desempenho muito superior. Isto indica que limitar demasiado a expansão da árvore pode excluir jogadas importantes. Embora `max_children` reduza o custo computacional, neste caso a perda estratégica foi grande.

**`max_depth=10` vs sem limite.**  
A profundidade limitada ficou pior que a versão sem limite, mas não de forma tão extrema como `max_children=3`. Isto sugere que limitar profundidade pode ser aceitável quando queremos reduzir tempo de decisão, desde que o limite não seja demasiado baixo.

**`c=1.41` vs `c=3`.**  
A diferença foi pequena. O valor `c=3` explora mais; `c=1.41` é uma escolha clássica para UCT. O resultado indica que, neste projeto, a constante `c` afeta o comportamento, mas não foi tão decisiva quanto `pure_mode` ou `max_children`.

**`best_tree_all` vs `best_tree_win`.**  
A árvore treinada apenas com jogadas de vencedores teve desempenho muito superior no confronto direto. A interpretação é que filtrar jogadas vencedoras reduziu ruído no dataset e produziu uma política mais consistente.


In [ ]:
def resumir_torneio(caminho_csv):
    df_torneio = pd.read_csv(caminho_csv)
    resumo = (
        df_torneio["Vencedor"]
        .value_counts()
        .rename_axis("Agente")
        .reset_index(name="Vitorias")
    )
    resumo["Ficheiro"] = caminho_csv.name
    resumo["Partidas"] = len(df_torneio)
    resumo["Jogadas Medias"] = round(df_torneio["Total_Jogadas"].mean(), 2)
    return resumo

resumos_torneios = []
for caminho in [RESULTS_DIR / "torneio.csv", RESULTS_DIR / "torneio_2.csv"]:
    if caminho.exists():
        resumos_torneios.append(resumir_torneio(caminho))

if resumos_torneios:
    resumo_torneios = pd.concat(resumos_torneios, ignore_index=True)
    display(resumo_torneios.head(20))
else:
    print("Nenhum ficheiro de torneio encontrado.")


### Leitura dos Torneios Gerais

Os torneios gerais colocam várias árvores e várias configurações de MCTS a jogar entre si. Eles não provam qual agente é melhor em absoluto, porque o resultado depende dos duelos incluídos, da ordem das partidas e do número de jogos. Mesmo assim, são úteis para observar tendências.

O padrão observado é:

- `mcts_best` tende a ser muito competitivo, confirmando o valor do MCTS otimizado.
- `best_tree_win` é a árvore mais forte e chega a competir bem com algumas configurações de MCTS.
- `worst_tree_all` e `worst_tree_win` ficam atrás, mostrando que a qualidade do dataset usado para treinar a árvore é decisiva.

A conclusão principal é que a árvore de decisão consegue imitar parte do comportamento do MCTS, mas a qualidade final depende fortemente dos exemplos usados no treino.


### Motor de Torneio para Novos Benchmarks

A secção seguinte define um motor de torneio para gerar novos ficheiros de benchmark. A execução fica controlada por uma variável booleana, porque partidas MCTS com muitas iterações podem demorar bastante.

A arena alterna quem começa para reduzir enviesamento do primeiro jogador e grava os resultados em `resultados/`.


In [ ]:
TREES = {
    "tree_best_all": "best_tree_all",
    "tree_best_win": "best_tree_win",
    "tree_worst_all": "worst_tree_all",
    "tree_worst_win": "worst_tree_win",
}

MCTS_TOURNAMENT_CONFIGS = {
    "mcts_best": dict(iterations=1000, c=1.41, max_children=None, max_depth=None, pure_mode=False),
    "mcts_pure": dict(iterations=1000, c=1.41, max_children=None, max_depth=None, pure_mode=True),
    "mcts_children_4": dict(iterations=1000, c=1.41, max_children=4, max_depth=None, pure_mode=True),
    "mcts_depth_15": dict(iterations=1000, c=1.41, max_children=None, max_depth=15, pure_mode=True),
    "mcts_high_c": dict(iterations=1000, c=3.0, max_children=None, max_depth=None, pure_mode=True),
    "mcts_low_c": dict(iterations=1000, c=1.0, max_children=None, max_depth=None, pure_mode=True),
}


def carregar_agentes_torneio():
    agentes = {}

    for alias, nome_ficheiro in TREES.items():
        caminho = MODELS_DIR / f"{nome_ficheiro}.pkl"
        if caminho.exists():
            with open(caminho, "rb") as f:
                agentes[alias] = pickle.load(f)
        else:
            print(f"[AVISO] árvore não encontrada: {caminho}")

    for alias, params in MCTS_TOURNAMENT_CONFIGS.items():
        agentes[alias] = MCTS(**params)

    return agentes


def run_matchup(num_games, ia_1, ia_2, name_ia1, name_ia2, output_file, max_turns=100):
    rows = []
    start_time = time.time()

    for i in range(num_games):
        game = PopOutGame()
        num_moves = 0
        ia_1_starts = (i % 2 == 0)

        if ia_1_starts:
            agent_p1, agent_p2 = ia_1, ia_2
            starter = name_ia1
        else:
            agent_p1, agent_p2 = ia_2, ia_1
            starter = name_ia2

        while True:
            if game.check_repetition() or num_moves > max_turns:
                winner_player = 0
                break

            current_agent = agent_p1 if game.current_player == PLAYER1 else agent_p2
            move = _get_ai_move(current_agent, game)

            if move is None:
                winner_player = 0
                break

            col, move_type = move
            if move_type == "d":
                game.drop_piece(col, game.current_player)
            else:
                game.pop_piece(col, game.current_player)

            num_moves += 1
            winner_now = game.check_winner_after_move(game.current_player)
            if winner_now is not None:
                winner_player = winner_now
                break

            game.current_player = PLAYER2 if game.current_player == PLAYER1 else PLAYER1

        if winner_player == 0:
            winner_name = "Empate"
        elif (winner_player == PLAYER1 and ia_1_starts) or (winner_player == PLAYER2 and not ia_1_starts):
            winner_name = name_ia1
        else:
            winner_name = name_ia2

        rows.append({
            "P1": name_ia1,
            "P2": name_ia2,
            "Vencedor": winner_name,
            "Quem_Comecou": starter,
            "Total_Jogadas": num_moves,
        })

    df_result = pd.DataFrame(rows)
    RESULTS_DIR.mkdir(exist_ok=True)
    output_path = RESULTS_DIR / output_file
    df_result.to_csv(output_path, index=False)

    elapsed = time.time() - start_time
    print(f"Resultado guardado em: {output_path}")
    print(f"Tempo total: {elapsed:.2f}s")
    return df_result


def plotar_benchmark(caminho_csv):
    df_result = pd.read_csv(caminho_csv)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    ax = sns.countplot(data=df_result, x="Vencedor", ax=axes[0], palette="viridis")
    axes[0].set_title("Taxa de vitórias global")
    axes[0].set_xlabel("Resultado / vencedor")
    axes[0].set_ylabel("Número de partidas")
    axes[0].tick_params(axis="x", rotation=20)
    for container in ax.containers:
        ax.bar_label(container)

    sns.histplot(data=df_result, x="Total_Jogadas", bins=15, ax=axes[1], color="#55A868")
    axes[1].set_title("Distribuição da duração das partidas")
    axes[1].set_xlabel("Total de jogadas")

    plt.tight_layout()
    plt.show()
    return df_result

RUN_TOURNAMENT = False

if RUN_TOURNAMENT:
    agentes_torneio = carregar_agentes_torneio()
    resultado_torneio = run_matchup(
        num_games=10,
        ia_1=agentes_torneio["mcts_best"],
        ia_2=agentes_torneio["mcts_pure"],
        name_ia1="mcts_best",
        name_ia2="mcts_pure",
        output_file="benchmark_mcts_best_vs_mcts_pure.csv",
    )
else:
    print("Arena de torneio não executada nesta célula. Defina RUN_TOURNAMENT = True para gerar novos benchmarks.")

# Exemplo de gráfico a partir de um CSV já existente:
# plotar_benchmark(RESULTS_DIR / "benchmark_mcts1000_c1.41_mcAll_dnone_vs_mcts1000_c3_mcAll_dnone.csv")


## 13. Interface de Jogo

A aplicação interativa pode ser iniciada pelo ficheiro `play.py`. A interface permite escolher entre:

1. humano vs humano;
2. humano vs IA;
3. IA vs IA.

Exemplo de execução no terminal:

```bash
python play.py
```

**Decisão de interface:** optámos por uma interface de terminal porque o foco do trabalho é inteligência artificial, não desenvolvimento gráfico. A interface textual é suficiente para demonstrar regras, alternância de turnos, jogadas humanas e jogadas de agentes.

A célula abaixo fica comentada porque chama um menu interativo, o que não é adequado para execução automática do notebook.

In [ ]:
# from src.game import main_menu
# main_menu(mcts_configs=mcts_configs, arvores=arvores)

## 14. Discussão dos Resultados

Os resultados devem ser interpretados considerando três pontos:

1. **Qualidade do MCTS:** mais iterações tendem a melhorar a decisão, mas aumentam o tempo por jogada.
2. **Qualidade do dataset:** a árvore só aprende tão bem quanto os exemplos gerados pelo MCTS.
3. **Conflitos no dataset:** o mesmo estado pode ter mais do que uma jogada aceitável, o que introduz ambiguidade para o ID3.

**Interpretação do MCTS otimizado:** quando `pure_mode=False`, o agente incorpora heurísticas de vitória e bloqueio iminente. Esta configuração costuma ser mais forte na prática, mas deve ser comparada com `pure_mode=True` para separar o efeito da pesquisa Monte Carlo do efeito das regras táticas adicionadas.

**Interpretação da árvore:** a árvore de decisão tem a vantagem de decidir rapidamente após o treino, mas normalmente perde capacidade tática face ao MCTS. Isto acontece porque a árvore transforma uma política de pesquisa num conjunto fixo de regras. Ela não simula consequências futuras; apenas reconhece padrões aprendidos no dataset.

**Conclusão técnica:** o MCTS é mais flexível durante o jogo, mas custa mais tempo de decisão. A árvore é mais rápida, mas depende fortemente da qualidade dos dados e da cobertura dos estados de treino.

## 15. Limitações

As principais limitações da solução são:

- MCTS com muitas iterações pode ser lento para jogos interativos.
- O MCTS otimizado não é uma implementação puramente aleatória, pois usa heurísticas táticas de vitória e defesa imediata.
- O ID3 trabalha melhor com atributos categóricos; no PopOut, o estado vetorizado gera muitas combinações possíveis.
- A árvore depende diretamente da qualidade e diversidade dos jogos simulados.
- Estados raros ou não vistos no treino podem cair no fallback da classe maioritária.
- Empates por repetição e decisões de empate em tabuleiro cheio são difíceis de representar como uma simples jogada `col_tipo`.

Estas limitações não invalidam a solução, mas ajudam a interpretar os resultados de forma mais rigorosa. O objetivo não é afirmar que o agente resolve PopOut perfeitamente, mas demonstrar uma implementação coerente dos algoritmos pedidos e uma avaliação honesta das suas consequências.

## 16. Conclusão

O projeto implementa uma solução completa para PopOut com dois paradigmas de inteligência artificial:

- **MCTS**, usado como agente adversarial e como gerador de dados;
- **ID3**, usado para aprender regras de classificação a partir do Iris e de estados do PopOut.

A solução suporta os modos de jogo exigidos, gera datasets próprios, treina árvores de decisão sem bibliotecas automáticas de machine learning e compara agentes através de torneios automáticos.

A principal decisão de engenharia foi tratar o MCTS não apenas como agente final, mas também como fonte de conhecimento para a árvore. Esta escolha liga as duas partes do trabalho: a pesquisa adversarial produz exemplos, e o ID3 tenta transformar esses exemplos em regras interpretáveis.

A principal decisão experimental foi manter versões alternativas: MCTS puro vs MCTS com heurísticas, árvores treinadas com todas as jogadas vs apenas jogadas vencedoras, e diferentes configurações de exploração. Essa comparação é essencial porque a proposta pede análise de alternativas, não apenas uma implementação única.

## 17. Como Executar

Instalar dependências:

```bash
pip install -r requirements.txt
```

Abrir o notebook:

```bash
jupyter notebook main_envio_proposta.ipynb
```

Executar a interface interativa:

```bash
python play.py
```